# Exp F - Cota Assintótica para rho_min(k)

**T. Bandeira - Junho de 2026**

Analisa o comportamento assintotico de rho_min(k) e em que escala o limiar rho* = 1e-6 se torna insuficiente.

- **F1** - Tabela extendida k=2..11 com cota estrutural
- **F2** - Ajuste assintotico e extrapolacao
- **F3** - Estrutura dos primos que realizam rho_min
- **F4** - Cota estrutural vs cota de Baker
- **F5** - Limiar adaptativo rho*(k)

In [1]:
from math import log, floor, exp
from sympy import isprime, primerange
import numpy as np

def e_max(p, k):
    return max(1, int(floor(k * log(2) / log(p))))

def build_seed(k):
    return list(primerange(2, 2**k))

def rho_adapt(m, k, seed):
    logm = log(m)
    best = 1.0
    for p in seed:
        emax = e_max(p, k)
        logp = log(p)
        for e in range(1, emax + 1):
            if m % (p**e) == 0:
                return 0.0
            d = abs(logm - e * logp) / logm
            if d < best:
                best = d
    for i, p1 in enumerate(seed):
        for p2 in seed[i:]:
            emax1, emax2 = e_max(p1, k), e_max(p2, k)
            logp1, logp2 = log(p1), log(p2)
            for e1 in range(1, emax1 + 1):
                for e2 in range(1, emax2 + 1):
                    val = e1*logp1 + e2*logp2
                    if val > logm * 1.05: break
                    d = abs(logm - val) / logm
                    if d < best: best = d
    return best

print("ok")


ok


## F1 - Tabela extendida e cota estrutural

Cota: rho_min(k) ~ 1 / (2^(k+1) * (k+1) * log2)

In [2]:
rho_mins = {}
for k in range(2, 12):
    seed = build_seed(k)
    primos = [m for m in range(2**k, 2**(k+1)) if isprime(m)]
    vals = sorted((rho_adapt(q, k, seed), q) for q in primos)
    rho_mins[k] = vals[0]
ks = sorted(rho_mins.keys())

print(f"{'k':>4} {'2^k':>8} {'rho_min':>14} {'primo':>8} {'cota 1/(2^(k+1)*(k+1)*log2)':>30} {'razao':>8}")
print("-" * 80)
for k in ks:
    r, q = rho_mins[k]
    cota = 1.0 / (2**(k+1) * (k+1) * log(2))
    print(f"{k:>4} {2**k:>8} {r:>14.4e} {q:>8} {cota:>30.4e} {r/cota:>8.4f}")


   k      2^k        rho_min    primo    cota 1/(2^(k+1)*(k+1)*log2)    razao
--------------------------------------------------------------------------------
   2        4     7.9218e-02        7                     6.0112e-02   1.3178
   3        8     2.8893e-02       13                     2.2542e-02   1.2817
   4       16     9.2454e-03       31                     9.0168e-03   1.0254
   5       32     3.9555e-03       61                     3.7570e-03   1.0528
   6       64     1.6191e-03      127                     1.6102e-03   1.0056
   7      128     7.2248e-04      251                     7.0444e-04   1.0256
   8      256     3.1554e-04      509                     3.1308e-04   1.0078
   9      512     1.4175e-04     1019                     1.4089e-04   1.0061
  10     1024     6.4376e-05     2039                     6.4040e-05   1.0052
  11     2048     2.9492e-05     4079                     2.9352e-05   1.0048


## F2 - Ajuste assintotico e extrapolacao

In [3]:
rmin_vals = np.array([rho_mins[k][0] for k in ks])
ks_arr = np.array(ks, dtype=float)
alpha_neg, log_C = np.polyfit(np.log(ks_arr), np.log(rmin_vals), 1)
C_fit = np.exp(log_C)
print(f"Ajuste log-log: rho_min(k) ~ {C_fit:.4f} / k^{-alpha_neg:.3f}")
print()
print("Extrapolacao - quando rho_min(k) ~ rho*:")
for rstar in [1e-6, 1e-8, 1e-10, 1e-15]:
    for k_test in range(1, 300):
        if 2**(k_test+1) * (k_test+1) > 1.0 / (rstar * log(2)):
            print(f"  rho* = {rstar:.0e}  -> k ~ {k_test}  (primos ~ 2^{k_test} ~ 10^{k_test*log(2)/log(10):.1f})")
            break


Ajuste log-log: rho_min(k) ~ 4.3501 / k^4.666

Extrapolacao - quando rho_min(k) ~ rho*:
  rho* = 1e-06  -> k ~ 16  (primos ~ 2^16 ~ 10^4.8)
  rho* = 1e-08  -> k ~ 22  (primos ~ 2^22 ~ 10^6.6)
  rho* = 1e-10  -> k ~ 28  (primos ~ 2^28 ~ 10^8.4)
  rho* = 1e-15  -> k ~ 44  (primos ~ 2^44 ~ 10^13.2)


## F3 - Estrutura dos primos que realizam rho_min

O primo que realiza rho_min e sempre o maior do bloco, e seu vizinho composto e q+-1.

In [4]:
from collections import Counter

def fatora(n, lim=50):
    fatores = []
    for p in range(2, min(n+1, lim)):
        while n % p == 0:
            fatores.append(p); n //= p
    if n > 1: fatores.append(n)
    c = Counter(fatores)
    return "*".join(f"{p}^{e}" if e > 1 else str(p) for p, e in sorted(c.items()))

print(f"{'k':>4} {'q(rho_min)':>12} {'maior primo A_k':>16} {'m* vizinho':>12} {'|q-m*|':>8} {'fat(m*)':>18}")
print("-" * 78)
for k in ks:
    r, q = rho_mins[k]
    q_top = 2**(k+1) - 1
    while not isprime(q_top): q_top -= 1
    mstar = q + 1 if not isprime(q+1) else q - 1
    mesmo = "ok" if q == q_top else f"top={q_top}"
    print(f"{k:>4} {q:>12} {q_top:>16} ({mesmo})  {mstar:>10} {abs(q-mstar):>8}  {fatora(mstar):>18}")


   k   q(rho_min)  maior primo A_k   m* vizinho   |q-m*|            fat(m*)
------------------------------------------------------------------------------
   2            7                7 (ok)           8        1                 2^3
   3           13               13 (ok)          14        1                 2*7
   4           31               31 (ok)          32        1                 2^5
   5           61               61 (ok)          62        1                2*31
   6          127              127 (ok)         128        1                 2^7
   7          251              251 (ok)         252        1           2^2*3^2*7
   8          509              509 (ok)         510        1            2*3*5*17
   9         1019             1021 (top=1021)        1020        1          2^2*3*5*17
  10         2039             2039 (ok)        2040        1          2^3*3*5*17
  11         4079             4093 (top=4093)        4080        1          2^4*3*5*17


## F4 - Cota estrutural (tight) vs cota de Baker (frouxa)

Comparacao entre as duas abordagens para cotar rho_min(k).

In [5]:
C_baker = 18
print(f"{'k':>4} {'rho_min':>14} {'cota estrutural':>16} {'r E':>8} {'cota Baker':>18} {'r B':>12}")
print("-" * 78)
for k in ks:
    obs = rho_mins[k][0]
    q_max = 2**(k+1) - 1
    while not isprime(q_max): q_max -= 1
    cota_e = 1.0 / (q_max * log(q_max))
    log_lam = -C_baker * log(q_max) * log(2) * log(max(k, 2))
    cota_b = exp(log_lam) / log(q_max)
    print(f"{k:>4} {obs:>14.4e} {cota_e:>16.4e} {obs/cota_e:>8.4f} {cota_b:>18.4e} {obs/cota_b:>12.2e}")
print()
print("Cota estrutural: tight, razao -> 1 para k grande.")
print("Cota Baker: frouxa por 10^4 a 10^95 - Baker garante positividade mas nao informa escala.")


   k        rho_min  cota estrutural      r E         cota Baker          r B
------------------------------------------------------------------------------
   2     7.9218e-02       7.3414e-02   1.0791         2.5255e-08     3.14e+06
   3     2.8893e-02       2.9990e-02   0.9634         2.0994e-16     1.38e+14
   4     9.2454e-03       9.3938e-03   0.9842         4.6682e-27     1.98e+24
   5     3.9555e-03       3.9878e-03   0.9919         3.4351e-37     1.15e+34
   6     1.6191e-03       1.6255e-03   0.9961         1.9227e-48     8.42e+44
   7     7.2248e-04       7.2104e-04   1.0020         9.9381e-60     7.27e+55
   8     3.1554e-04       3.1523e-04   1.0010         9.5720e-72     3.30e+67
   9     1.4175e-04       1.4136e-04   1.0027         4.6766e-84     3.03e+79
  10     6.4376e-05       6.4360e-05   1.0002         1.1048e-96     5.83e+91
  11     2.9492e-05       2.9376e-05   1.0040        1.0377e-109    2.84e+104

Cota estrutural: tight, razao -> 1 para k grande.
Cota Baker: 

## F5 - Limiar adaptativo rho*(k)

Define rho*(k) = c / (2^(k+1) * (k+1) * log2) com c=0.1 como substituto do limiar fixo para k grande.

In [6]:
c = 0.1
print(f"Limiar adaptativo: rho*(k) = {c} / (2^(k+1) * (k+1) * log2)")
print()
print(f"{'k':>4} {'rho_min obs':>14} {'rho*(k) c=0.1':>16} {'margem':>12}")
print("-" * 50)
for k in ks:
    obs = rho_mins[k][0]
    rstar_adapt = c / (2**(k+1) * (k+1) * log(2))
    print(f"{k:>4} {obs:>14.4e} {rstar_adapt:>16.4e} {obs/rstar_adapt:>10.1f}x")
print()
print("Com c=0.1: margem de aprox 10x acima de rho_min em todos os blocos.")
print("Implementacao: rho_star = 0.1 / (2**(k+1) * (k+1) * log(2))")


Limiar adaptativo: rho*(k) = 0.1 / (2^(k+1) * (k+1) * log2)

   k    rho_min obs    rho*(k) c=0.1       margem
--------------------------------------------------
   2     7.9218e-02       6.0112e-03       13.2x
   3     2.8893e-02       2.2542e-03       12.8x
   4     9.2454e-03       9.0168e-04       10.3x
   5     3.9555e-03       3.7570e-04       10.5x
   6     1.6191e-03       1.6102e-04       10.1x
   7     7.2248e-04       7.0444e-05       10.3x
   8     3.1554e-04       3.1308e-05       10.1x
   9     1.4175e-04       1.4089e-05       10.1x
  10     6.4376e-05       6.4040e-06       10.1x
  11     2.9492e-05       2.9352e-06       10.0x

Com c=0.1: margem de aprox 10x acima de rho_min em todos os blocos.
Implementacao: rho_star = 0.1 / (2**(k+1) * (k+1) * log(2))


## Resumo

**Resultado principal:** rho_min(k) ~ 1 / (q_max * log q_max) ~ 1 / (2^(k+1) * (k+1) * log2), com razao observado/cota -> 1 para k >= 6.

**Limiar atual rho* = 1e-6 e seguro para k <= 15 (primos ate ~32768).**

Para escalas maiores: rho*(k) = c / (2^(k+1) * (k+1) * log2) com c = 0.1.

**Por que Baker nao resolve:** Baker garante positividade mas e frouxa por 10^20+ para esses valores. O problema e de gaps de primos, nao de teoria transcendente.